# Solution 2.2.1 — Loading Data & First Diagnostics

### Path Setup

In [ ]:
import os
import pandas as pd

DATA_RAW_DIR = '../../../data/0_raw'
FILE_NAME = 'datania_households_raw.csv'
raw_path = os.path.join(DATA_RAW_DIR, FILE_NAME)

print('Data path:', raw_path)
print('Exists?:', os.path.exists(raw_path))

Data path: ../../../data/0_raw/tanzania/datania_households_raw.csv
Exists?: True


---

## Task 1 — Load the data and take a first look

In [3]:
df = pd.read_csv(raw_path)

df.head()

,hh_id,region_code,province_name,district,urban_rural,hh_size,income_dkw,survey_date,pop_density,education_code,age
0,HH0001,1.0,Eastern Province,Kurtosis Bay,Urban,4,45 000,2025-01-10,780,3,42
1,HH0002,2.0,Northern Province,Vector Hills,Rural,6,Ar 32000,2025-01-11,120,2,39
2,HH0003,3.0,Central Province,Polaris District,Urban,3,54000,2025-01-12,640,4,33
3,HH0004,4.0,Southern Province,Lagoon Point,Rural,5,NaN,2025-01-13,80,1,51
4,HH0005,5.0,Western Province,Gamma Plains,Urban,2,unknown,2025-01-13,520,2,28


In [4]:
df.tail()

,hh_id,region_code,province_name,district,urban_rural,hh_size,income_dkw,survey_date,pop_density,education_code,age
23,HH0023,NaN,Western Province,Chi Meadow,Rural,2,35000,2025-01-31,105,1,58
24,HH0024,6.0,Highland Province,Psi Canal,Urban,3,Ar 99000,2025-02-01,640,4,32
25,HH0025,5.0,Western Province,Omega Town,Rural,4,61000,2025-02-02,190,2,999
26,HH0005,5.0,Western Province,Gamma Plains,Urban,2,58000,2025-02-05,520,2,28
27,HH0026,99.0,Northern Province,Vector Hills,Rural,3,39000,2025-02-07,9999,2,40


In [5]:
df.sample(5)

,hh_id,region_code,province_name,district,urban_rural,hh_size,income_dkw,survey_date,pop_density,education_code,age
8,HH0009,3.0,Central Province,Omega Town,Urban,8,"1,200,000",2025-01-17,840,3,37
18,HH0018,6.0,Highland Province,Pi Coast,Urban,6,94000,2025-01-26,680,3,44
9,HH0010,4.0,Southern Province,Kappa Port,Urbn,4,41000,2025/01/18,710,2,45
19,HH0019,1.0,Eastern Province,Rho Basin,Rural,3,33000,2025-01-27,130,2,38
23,HH0023,NaN,Western Province,Chi Meadow,Rural,2,35000,2025-01-31,105,1,58


**Answers:**

- Headers loaded correctly: `hh_id`, `region_code`, `province_name`, `district`, `urban_rural`, `hh_size`, `income_dkw`, `survey_date`, `pop_density`, `education_code`, `age`.
- Suspicious values visible from a quick look: `income_dkw` contains values like `Ar 32000`, `45 000`, `unknown`; `urban_rural` has `Urbn`; `hh_size` has `0` and `-1`; `survey_date` has at least two different formats.
- A visual inspection is not sufficient. It only shows a subset of rows. Systematic diagnostics are needed to catch issues that don't appear in the first or last 5 rows.

---

## Task 2 — Check the structure with `df.info()` and `isna().sum()`

In [6]:
df.info()

df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   hh_id           28 non-null     str    
 1   region_code     27 non-null     float64
 2   province_name   27 non-null     str    
 3   district        28 non-null     str    
 4   urban_rural     28 non-null     str    
 5   hh_size         28 non-null     int64  
 6   income_dkw      25 non-null     str    
 7   survey_date     27 non-null     str    
 8   pop_density     28 non-null     int64  
 9   education_code  28 non-null     int64  
 10  age             28 non-null     int64  
dtypes: float64(1), int64(4), str(6)
memory usage: 4.0 KB


hh_id             0
region_code       1
province_name     1
district          0
urban_rural       0
hh_size           0
income_dkw        3
survey_date       1
pop_density       0
education_code    0
age               0
dtype: int64

**Answers:**

- `income_dkw` is `str` (text) because it contains non-numeric characters like `Ar`, spaces, and the word `unknown`. pandas falls back to text when it cannot parse the column as a number.
- `survey_date` is `str` — it cannot be used for date arithmetic until converted with `pd.to_datetime()`.
- `region_code` is inferred as `int64` by default, silently dropping leading zeros (e.g. `01` → `1`).
- Columns with missing values: `province_name` and `region_code` each have at least one `NaN`.

---

## Task 3 — Why loading as string matters (leading zeros)

In [7]:
df_default = pd.read_csv(raw_path)
print('Default dtypes:')
print(df_default[['hh_id', 'region_code']].dtypes)
print()
print(df_default[['hh_id', 'region_code']].head(8))

Default dtypes:
hh_id              str
region_code    float64
dtype: object

    hh_id  region_code
0  HH0001          1.0
1  HH0002          2.0
2  HH0003          3.0
3  HH0004          4.0
4  HH0005          5.0
5  HH0006          6.0
6  HH0007          1.0
7  HH0008          2.0


In [8]:
df = pd.read_csv(
    raw_path,
    dtype={'hh_id': str, 'region_code': str}
)
print('Type-safe dtypes:')
print(df[['hh_id', 'region_code']].dtypes)
print()
print(df[['hh_id', 'region_code']].head(8))

Type-safe dtypes:
hh_id          str
region_code    str
dtype: object

    hh_id region_code
0  HH0001          01
1  HH0002          02
2  HH0003          03
3  HH0004          04
4  HH0005          05
5  HH0006          06
6  HH0007          01
7  HH0008          02


**Answers:**

- With default loading, `region_code` is `int64`: `'01'` becomes `1`, `'02'` becomes `2`. With `dtype=str` it stays `'01'`, `'02'`.
- Identifier columns should always be text because they are labels, not quantities. You never add or average them, and their formatting (leading zeros, dashes) is meaningful.
- Silent failures if leading zeros are dropped: merges with geographic lookup tables that use zero-padded codes will find no matches; `value_counts()` will show `1` and `01` as different values after a partial fix.

---

## Task 4 — Summary statistics with `describe()`

In [9]:
df.describe()

,hh_size,pop_density,education_code,age
count,28.000000,28.000000,28.000000,28.000000
mean,6.928571,755.857143,5.821429,75.035714
std,18.149438,1832.523209,18.285905,181.433486
min,-1.000000,60.000000,1.000000,24.000000
25%,2.750000,127.500000,2.000000,31.750000
50%,3.500000,525.000000,2.000000,39.500000
75%,5.000000,640.000000,3.000000,49.500000
max,99.000000,9999.000000,99.000000,999.000000


In [10]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
hh_id,28,26,HH0005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region_code,27,8,05,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
province_name,27,6,Western Province,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
district,28,23,Vector Hills,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
urban_rural,28,3,Urban,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hh_size,28.0,NaN,NaN,NaN,6.928571,18.149438,-1.0,2.75,3.5,5.0,99.0
income_dkw,25,24,65000,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
survey_date,27,25,2025-01-13,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pop_density,28.0,NaN,NaN,NaN,755.857143,1832.523209,60.0,127.5,525.0,640.0,9999.0
education_code,28.0,NaN,NaN,NaN,5.821429,18.285905,1.0,2.0,2.0,3.0,99.0


**Answers:**

- `hh_size` has a `min` of `-1` and a `max` of `99` — both impossible for a real household. `age` has a `max` of `999`, clearly a sentinel value.
- `income_dkw` does not appear in the numeric summary because its dtype is `object` (text). This confirms it needs cleaning before any numeric analysis.
- For text columns: `urban_rural` likely has a suspicious `unique` count if `'Urbn'` is present. `region_code` shows both `'1'` and `'01'` as separate categories.
- `count` below the total row count indicates missing values — compare each column's `count` to `df.shape[0]`.

---

## Task 5 — Explore categorical columns with `value_counts()`

In [11]:
df['urban_rural'].value_counts()

urban_rural
Urban    14
Rural    13
Urbn      1
Name: count, dtype: int64

In [12]:
df['urban_rural'].value_counts(dropna=False)

urban_rural
Urban    14
Rural    13
Urbn      1
Name: count, dtype: int64

In [13]:
df['region_code'].value_counts(dropna=False)

region_code
05     5
06     5
02     4
03     4
04     4
01     3
1      1
NaN    1
99     1
Name: count, dtype: int64

In [14]:
print('Number of distinct districts:', df['district'].nunique())
print(df['district'].unique())

Number of distinct districts: 23
<ArrowStringArray>
[    'Kurtosis Bay',     'Vector Hills', 'Polaris District',
     'Lagoon Point',     'Gamma Plains',      'Sigma Ridge',
      'North Delta',      'South Delta',       'Omega Town',
       'Kappa Port',      'Beta Valley',     'Lambda Falls',
      'Mu Crossing',       'Nu Station',        'Xi Market',
     'Omicron Yard',         'Pi Coast',        'Rho Basin',
      'Tau Heights',    'Upsilon Grove',        'Phi Point',
       'Chi Meadow',        'Psi Canal']
Length: 23, dtype: str


**Answers:**

- `urban_rural` contains `'Urbn'` — a typo for `'Urban'`. It appears as a separate category.
- `region_code` has `'99'` (a coded missing value) and `'1'` alongside `'01'` — inconsistent zero-padding.
- `dropna=False` is important because without it, missing values are silently excluded from the count. A column that appears complete may in fact have `NaN` entries that only show up when `dropna=False` is used.

---

## Task 6 — Inspect column names and rename if needed

In [15]:
df.columns

Index(['hh_id', 'region_code', 'province_name', 'district', 'urban_rural',
       'hh_size', 'income_dkw', 'survey_date', 'pop_density', 'education_code',
       'age'],
      dtype='str')

In [16]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(' ', '_', regex=False)
)

print(df.columns.tolist())

['hh_id', 'region_code', 'province_name', 'district', 'urban_rural', 'hh_size', 'income_dkw', 'survey_date', 'pop_density', 'education_code', 'age']


**Answer:** No column names changed here because they were already lowercase with underscores. This step matters most when data arrives from Excel files or other tools that use spaces, mixed casing, or special characters in headers — which is very common in practice.

---

## Task 7 — Load only the columns you need with `usecols`

In [17]:
COLS = [
    'hh_id', 'region_code', 'province_name', 'district',
    'urban_rural', 'hh_size', 'income_dkw', 'survey_date',
    'pop_density', 'education_code', 'age'
]

df_small = pd.read_csv(raw_path, usecols=COLS)

df_small.head()

,hh_id,region_code,province_name,district,urban_rural,hh_size,income_dkw,survey_date,pop_density,education_code,age
0,HH0001,1.0,Eastern Province,Kurtosis Bay,Urban,4,45 000,2025-01-10,780,3,42
1,HH0002,2.0,Northern Province,Vector Hills,Rural,6,Ar 32000,2025-01-11,120,2,39
2,HH0003,3.0,Central Province,Polaris District,Urban,3,54000,2025-01-12,640,4,33
3,HH0004,4.0,Southern Province,Lagoon Point,Rural,5,NaN,2025-01-13,80,1,51
4,HH0005,5.0,Western Province,Gamma Plains,Urban,2,unknown,2025-01-13,520,2,28


In [18]:
print(f'Full DataFrame:  {df.shape}')
print(f'Subset:          {df_small.shape}')
print(f'Memory full:     {df.memory_usage(deep=True).sum() / 1e6:.3f} MB')
print(f'Memory subset:   {df_small.memory_usage(deep=True).sum() / 1e6:.3f} MB')

Full DataFrame:  (28, 11)
Subset:          (28, 11)
Memory full:     0.004 MB
Memory subset:   0.004 MB


**Answers:**

- Memory savings are modest on this small dataset but scale significantly with wider files (hundreds of columns is common in survey data).
- A misspelled column name in `usecols` raises a `ValueError` immediately at load time — useful because it fails fast rather than silently producing a wrong result.
- `pd.read_csv(usecols=COLS)` never reads the excluded columns into memory. `df[COLS]` after a full load reads everything first, then discards it. For large files the difference in peak memory and load time is substantial.